# 18: LSTM & GRU - Solving the Memory Problem

## Selective Memory

RNNs forget too easily (vanishing gradients). **LSTM (Long Short-Term Memory)** and **GRU (Gated Recurrent Unit)** solve this with **gates**:
- Decide what to remember
- Decide what to forget
- Decide what to output

### The Web Dev Analogy

LSTMs are like **smart caching**:
- **Forget gate**: Clear old cache entries
- **Input gate**: Add new cache entries
- **Output gate**: Decide what to expose
- Long-term storage (cell state) + working memory (hidden state)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

print("Ready to explore LSTM and GRU! 🧠")

## 1. The LSTM Architecture

LSTM has **two states**:
1. **Cell state (c_t)**: Long-term memory (highway for gradients!)
2. **Hidden state (h_t)**: Short-term working memory

And **three gates** (each is a mini neural network):
1. **Forget gate**: What to remove from cell state
2. **Input gate**: What new info to add to cell state
3. **Output gate**: What to output from cell state

In [ ]:
# LSTM formulas (don't memorize, just understand the idea!)
print("LSTM Gates:")
print("=" * 70)
print("\n1. Forget Gate (what to forget from cell state):")
print("   f_t = sigmoid(W_f * [h_(t-1), x_t] + b_f)")
print("   → Values close to 0 = forget, close to 1 = keep")

print("\n2. Input Gate (what new info to add):")
print("   i_t = sigmoid(W_i * [h_(t-1), x_t] + b_i)")
print("   c~_t = tanh(W_c * [h_(t-1), x_t] + b_c)  (candidate values)")
print("   → Decide what to add and how much")

print("\n3. Update Cell State:")
print("   c_t = f_t * c_(t-1) + i_t * c~_t")
print("   → Forget some + add some = new cell state")

print("\n4. Output Gate (what to output):")
print("   o_t = sigmoid(W_o * [h_(t-1), x_t] + b_o)")
print("   h_t = o_t * tanh(c_t)")
print("   → Filter cell state to produce hidden state")

print("\n💡 Key insight: Cell state has a 'highway' for gradients!")
print("   Additions/multiplications by gates preserve gradients.")

`★ Insight ─────────────────────────────────────`

**Why LSTM solves vanishing gradients:**
1. **Cell state highway** - gradients flow directly through addition
2. **Gates control flow** - protect against vanishing
3. **Forget gate can be close to 1** - preserves long-term memory

Think: Cell state = long-term storage, Hidden state = working memory

`─────────────────────────────────────────────────`

## 2. PyTorch LSTM

In [ ]:
# Create LSTM layer
input_size = 10
hidden_size = 20
num_layers = 1

lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

print(f"LSTM Architecture:")
print(f"  Input size: {input_size}")
print(f"  Hidden size: {hidden_size}")
print(f"  Num layers: {num_layers}")
print(f"  Parameters: {sum(p.numel() for p in lstm.parameters()):,}")

# Input: batch of sequences
batch_size = 2
seq_len = 5
x = torch.randn(batch_size, seq_len, input_size)

# Forward pass
output, (h_n, c_n) = lstm(x)

print(f"\nInput shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"  → Hidden states at every time step")

print(f"\nFinal hidden state: {h_n.shape}")
print(f"Final cell state: {c_n.shape}")
print(f"  → Both needed for continuing the sequence")

## 3. LSTM vs RNN: Long-Range Dependencies

In [ ]:
# Generate sequences where first element determines label
def generate_long_range(n_samples, seq_len):
    """First element determines label, rest is noise."""
    X = torch.randn(n_samples, seq_len, 1)
    y = (X[:, 0, 0] > 0).float()
    return X, y

# Classifiers
class RNNClassifier(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(1, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        _, h_n = self.rnn(x)
        return self.fc(h_n.squeeze(0))

class LSTMClassifier(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n.squeeze(0))

# Compare on different sequence lengths
seq_lengths = [10, 20, 50, 100]
rnn_results = []
lstm_results = []

print("Comparing RNN vs LSTM on long-range dependencies...")
print("=" * 70)

for seq_len in seq_lengths:
    print(f"\nSequence length: {seq_len}")
    
    # Generate data
    train_X, train_y = generate_long_range(1000, seq_len)
    test_X, test_y = generate_long_range(200, seq_len)
    
    criterion = nn.BCEWithLogitsLoss()
    
    # Train RNN
    rnn_model = RNNClassifier(hidden_size=32)
    rnn_opt = torch.optim.Adam(rnn_model.parameters(), lr=0.001)
    
    for epoch in range(50):
        outputs = rnn_model(train_X)
        loss = criterion(outputs, train_y.unsqueeze(1))
        loss.backward()
        rnn_opt.step()
        rnn_opt.zero_grad()
    
    # Test RNN
    rnn_model.train(False)
    with torch.no_grad():
        rnn_preds = (torch.sigmoid(rnn_model(test_X)) > 0.5).float()
        rnn_acc = (rnn_preds.squeeze() == test_y).float().mean().item()
    rnn_results.append(rnn_acc)
    
    # Train LSTM
    lstm_model = LSTMClassifier(hidden_size=32)
    lstm_opt = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
    
    for epoch in range(50):
        outputs = lstm_model(train_X)
        loss = criterion(outputs, train_y.unsqueeze(1))
        loss.backward()
        lstm_opt.step()
        lstm_opt.zero_grad()
    
    # Test LSTM
    lstm_model.train(False)
    with torch.no_grad():
        lstm_preds = (torch.sigmoid(lstm_model(test_X)) > 0.5).float()
        lstm_acc = (lstm_preds.squeeze() == test_y).float().mean().item()
    lstm_results.append(lstm_acc)
    
    print(f"  RNN Accuracy:  {rnn_acc:.2%}")
    print(f"  LSTM Accuracy: {lstm_acc:.2%}")

In [ ]:
# Visualize comparison
plt.figure(figsize=(12, 6))

plt.plot(seq_lengths, rnn_results, 'o-', linewidth=2, markersize=10, label='RNN')
plt.plot(seq_lengths, lstm_results, 's-', linewidth=2, markersize=10, label='LSTM')
plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random guessing')

plt.xlabel('Sequence Length', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('LSTM Maintains Performance on Long Sequences', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim([0.4, 1.05])

plt.tight_layout()
plt.show()

print("\n✅ LSTM handles long-range dependencies much better!")
print("   RNN degrades quickly, LSTM stays strong.")

## 4. GRU: Simplified LSTM

**GRU (Gated Recurrent Unit)** simplifies LSTM:
- Only **2 gates** (vs 3 in LSTM)
- Only **1 state** (vs 2 in LSTM)
- Fewer parameters, faster training
- Often performs similarly to LSTM

In [ ]:
print("GRU Gates:")
print("=" * 70)
print("\n1. Reset Gate (how much past to forget):")
print("   r_t = sigmoid(W_r * [h_(t-1), x_t])")

print("\n2. Update Gate (how much to update):")
print("   z_t = sigmoid(W_z * [h_(t-1), x_t])")

print("\n3. New hidden state:")
print("   h~_t = tanh(W * [r_t * h_(t-1), x_t])")
print("   h_t = (1 - z_t) * h_(t-1) + z_t * h~_t")

print("\n💡 Simpler than LSTM, but still effective!")

In [ ]:
# GRU in PyTorch
gru = nn.GRU(input_size=10, hidden_size=20, batch_first=True)

x = torch.randn(2, 5, 10)
output, h_n = gru(x)

print(f"GRU output shape: {output.shape}")
print(f"GRU hidden shape: {h_n.shape}")
print(f"\nNote: No cell state (c_n) - simpler than LSTM!")

# Compare parameters
lstm = nn.LSTM(10, 20, batch_first=True)
gru = nn.GRU(10, 20, batch_first=True)
rnn = nn.RNN(10, 20, batch_first=True)

print(f"\nParameter counts:")
print(f"  RNN:  {sum(p.numel() for p in rnn.parameters()):,}")
print(f"  GRU:  {sum(p.numel() for p in gru.parameters()):,}")
print(f"  LSTM: {sum(p.numel() for p in lstm.parameters()):,}")
print(f"\nLSTM has most parameters (4 gates), GRU is middle ground.")

## 5. Bidirectional LSTM/GRU

In [ ]:
# Sometimes we want to see the FUTURE too!
# Bidirectional: process sequence forward AND backward

bi_lstm = nn.LSTM(input_size=10, hidden_size=20, batch_first=True, bidirectional=True)

x = torch.randn(2, 5, 10)
output, (h_n, c_n) = bi_lstm(x)

print(f"Input shape: {x.shape}")
print(f"\nBidirectional LSTM output: {output.shape}")
print(f"  → Hidden size is 2x (forward + backward)")
print(f"  → Each time step sees past AND future context")

print(f"\nHidden state: {h_n.shape}")
print(f"  → (2, batch, hidden) - forward and backward")

print("\n💡 Use bidirectional when you have the full sequence!")
print("   (Not suitable for real-time generation)")

## 6. Practical Sentiment Classification

In [ ]:
# Simple sentiment model with LSTM
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)  # *2 for bidirectional
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, text):
        # text: (batch, seq_len)
        embedded = self.embedding(text)  # (batch, seq_len, embedding_dim)
        embedded = self.dropout(embedded)
        
        output, (hidden, cell) = self.lstm(embedded)
        # output: (batch, seq_len, hidden_dim*2)
        
        # Concatenate final forward and backward hidden states
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        # hidden: (batch, hidden_dim*2)
        
        hidden = self.dropout(hidden)
        output = self.fc(hidden)
        return output

# Create model
model = SentimentLSTM(
    vocab_size=5000,
    embedding_dim=100,
    hidden_dim=128,
    output_dim=1
)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test forward pass
sample_text = torch.randint(0, 5000, (4, 20))  # 4 sequences, length 20
sample_output = model(sample_text)
print(f"\nInput shape: {sample_text.shape}")
print(f"Output shape: {sample_output.shape}")
print(f"  → One sentiment score per sequence")

`★ Insight ─────────────────────────────────────`

**When to use what:**
1. **RNN**: Simple sequences, short dependencies, need speed
2. **LSTM**: Long sequences, complex dependencies, proven architecture
3. **GRU**: Middle ground - simpler than LSTM, better than RNN
4. **Bidirectional**: Classification tasks where you have full sequence

Modern trend: LSTMs/GRUs being replaced by Transformers for many tasks!

`─────────────────────────────────────────────────`

## 📝 Check Your Understanding

1. What are the three gates in an LSTM?
2. What's the difference between cell state and hidden state?
3. Why does LSTM solve the vanishing gradient problem?
4. How is GRU different from LSTM?
5. When should you use bidirectional RNN/LSTM?

## 🎯 Summary

**LSTM architecture**:
- **Cell state**: Long-term memory with gradient highway
- **Hidden state**: Short-term working memory
- **Gates**: Control information flow (forget, input, output)

**GRU architecture**:
- Simpler: 2 gates, 1 state
- Fewer parameters, faster training
- Often comparable performance to LSTM

**Key advantage**:
- Handle long-range dependencies
- Solve vanishing gradient problem
- Work well for sequence tasks

**PyTorch usage**:
- `nn.LSTM(input_size, hidden_size, bidirectional=True)`
- `nn.GRU(input_size, hidden_size)`
- Returns (output, (h_n, c_n)) for LSTM
- Returns (output, h_n) for GRU

**Next up**: Text generation with RNNs! →